## TD-Gammon

In [3]:
# board[i] > 0  means white has that many pieces on point i
# board[i] < 0  means black has that many pieces on point i
# board[i] = 0  means empty

board = [0] * 26

# index 0    = white's bar
# index 25   = black's bar
# index 1-24 = the 24 points

# Starting position
board = [0,                    # white bar
         -2, 0, 0, 0, 0, 5,   # points 1-6
          0, 3, 0, 0, 0, -5,  # points 7-12
          5, 0, 0, 0, -3, 0,  # points 13-18
         -5, 0, 0, 0, 0, 2,   # points 19-24
          0]                   # black bar

In [4]:
def flip_board(board):
    # Create a new board with the positions flipped
    new_board = [0] * 26 
    new_board[0] = board[25]
    new_board[25] = board[0]
    for i in range(1, 25):
        new_board[i] = -board[26 - i]
    return new_board

In [5]:
def encode_board(board, player, cube=1):
    inputs = []

    if player == "BLACK":
        board = flip_board(board)

    mine_on_board = 0
    opp_on_board  = 0

    for i in range(1, 25):
        n     = board[i]
        mine  = max(n, 0)
        inputs.append(1.0 if mine >= 1 else 0.0)
        inputs.append(1.0 if mine >= 2 else 0.0)
        inputs.append(1.0 if mine >= 3 else 0.0)
        inputs.append(max(mine - 3, 0) / 2.0)
        mine_on_board += mine

        theirs = max(-n, 0)
        inputs.append(1.0 if theirs >= 1 else 0.0)
        inputs.append(1.0 if theirs >= 2 else 0.0)
        inputs.append(1.0 if theirs >= 3 else 0.0)
        inputs.append(max(theirs - 3, 0) / 2.0)
        opp_on_board += theirs

    # board[0] = current player bar, board[25] = opponent bar (after flip)
    my_off  = 15 - mine_on_board - board[0]
    opp_off = 15 - opp_on_board  - board[25]

    inputs.append(board[0]  / 2.0)
    inputs.append(board[25] / 2.0)
    inputs.append(my_off    / 15.0)
    inputs.append(opp_off   / 15.0)
    inputs.append(1.0 if player == "WHITE" else 0.0)
    inputs.append(cube / 64.0)

    return inputs  # 198 floats

In [25]:
import numpy as np
import random
import time
import torch
import torch.nn as nn

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


def initial_board():
    return [0, -2, 0, 0, 0, 0, 5, 0, 3, 0, 0, 0, -5, 5, 0, 0, 0, -3, 0, -5, 0, 0, 0, 0, 2, 0]


def roll_dice():
    d1, d2 = random.randint(1, 6), random.randint(1, 6)
    return [d1, d2, d1, d2] if d1 == d2 else [d1, d2]


def game_outcome(board):
    """Returns (winner, outcome_type) or (None, None). outcome_type: 'win'/'gammon'/'backgammon'."""
    white_on = sum(board[i] for i in range(1, 25) if board[i] > 0)
    black_on = sum(-board[i] for i in range(1, 25) if board[i] < 0)
    white_off = 15 - white_on - board[0]
    black_off = 15 - black_on - board[25]

    if white_off == 15:
        winner = "WHITE"
        loser_off, loser_bar = black_off, board[25]
        loser_in_winner_home = any(board[i] < 0 for i in range(1, 7))
    elif black_off == 15:
        winner = "BLACK"
        loser_off, loser_bar = white_off, board[0]
        loser_in_winner_home = any(board[i] > 0 for i in range(19, 25))
    else:
        return None, None

    if loser_off == 0 and (loser_bar > 0 or loser_in_winner_home):
        return winner, "backgammon"
    if loser_off == 0:
        return winner, "gammon"
    return winner, "win"


def outcome_target(winner, outcome_type, current_player):
    """6-element terminal target from current_player's perspective.
    [p_win, p_gammon_win, p_backgammon_win, p_loss, p_gammon_loss, p_backgammon_loss]"""
    t = np.zeros(6, dtype=np.float32)
    if winner == current_player:
        t[{"win": 0, "gammon": 1, "backgammon": 2}[outcome_type]] = 1.0
    else:
        t[{"win": 3, "gammon": 4, "backgammon": 5}[outcome_type]] = 1.0
    return t


class TDGammon(nn.Module):
    def __init__(self, input_size=198, hidden_size=160, output_size=6, lr=0.01):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.Sigmoid(),
            nn.Linear(hidden_size, output_size),
            nn.Sigmoid()
        )
        self.to(device)
        self.optimizer = torch.optim.SGD(self.parameters(), lr=lr)

    def forward(self, x):
        return self.net(x)

    def td_update(self, v, delta):
        """
        v    : tensor output of forward(), must have gradient.
        delta: TD error — numpy array or tensor, treated as a constant (no grad).
        """
        if not isinstance(delta, torch.Tensor):
            delta = torch.tensor(delta, dtype=torch.float32, device=device)
        self.optimizer.zero_grad()
        loss = -(delta.detach() * v).sum()
        loss.backward()
        self.optimizer.step()


# ---------------------------------------------------------------------------
# Move generator — mirrors Swift MoveGenerator exactly.
# board[0]=white bar, board[25]=black bar, board[1-24]: +white/-black
# Move steps: (from, to, die)  |  -1=white bears off, 26=black bears off
# ---------------------------------------------------------------------------

def generate_legal_moves(board, dice, player):
    results = []
    _gen(board, list(dice), player, [], results)

    if not results:
        return []
    max_used = max(len(m) for m, _ in results)
    if max_used == 0:
        return []

    candidates = [(m, s) for m, s in results if len(m) == max_used]

    if max_used == 1 and len(dice) == 2 and dice[0] != dice[1]:
        higher = max(dice)
        with_higher = [(m, s) for m, s in candidates if m[0][2] == higher]
        if with_higher:
            candidates = with_higher

    seen, unique = set(), []
    for m, s in candidates:
        key = tuple(s)
        if key not in seen:
            seen.add(key)
            unique.append(m)
    return unique


def _gen(board, remaining, player, current, results):
    singles = _singles(board, remaining, player)
    if not singles:
        results.append((current, board)); return
    tried = set()
    for die in remaining:
        if die in tried: continue
        tried.add(die)
        for m in [s for s in singles if s[2] == die]:
            nxt = list(remaining); nxt.remove(die)
            _gen(_apply_step(board, m, player), nxt, player, current + [m], results)


def _singles(board, dice, player):
    moves = []
    for die in set(dice):
        if player == "WHITE":
            if board[0] > 0:
                to = 25 - die
                if board[to] > -2:
                    moves.append((0, to, die))
            else:
                for pt in range(1, 25):
                    if board[pt] <= 0: continue
                    dest = pt - die
                    if 1 <= dest <= 24:
                        if board[dest] > -2: moves.append((pt, dest, die))
                    elif _can_bear_off_white(board) and _bear_off_ok_white(pt, dest, board):
                        moves.append((pt, -1, die))
        else:
            if board[25] > 0:
                to = die
                if board[to] < 2:
                    moves.append((25, to, die))
            else:
                for pt in range(1, 25):
                    if board[pt] >= 0: continue
                    dest = pt + die
                    if 1 <= dest <= 24:
                        if board[dest] < 2: moves.append((pt, dest, die))
                    elif _can_bear_off_black(board) and _bear_off_ok_black(pt, dest, board):
                        moves.append((pt, 26, die))
    return moves


def _can_bear_off_white(board):
    return board[0] == 0 and all(board[i] <= 0 for i in range(7, 25))

def _can_bear_off_black(board):
    return board[25] == 0 and all(board[i] >= 0 for i in range(1, 19))

def _bear_off_ok_white(point, dest, board):
    if dest == 0: return True
    return all(board[p] <= 0 for p in range(point + 1, 7))

def _bear_off_ok_black(point, dest, board):
    if dest == 25: return True
    return all(board[p] >= 0 for p in range(19, point))


def _apply_step(board, step, player):
    board = list(board)
    frm, to, _ = step
    if player == "WHITE":
        if frm == 0: board[0] -= 1
        else:        board[frm] -= 1
        if to != -1:
            if board[to] == -1: board[to] = 0; board[25] += 1
            board[to] += 1
    else:
        if frm == 25: board[25] -= 1
        else:         board[frm] += 1
        if to != 26:
            if board[to] == 1: board[to] = 0; board[0] += 1
            board[to] -= 1
    return board


def apply_move(board, move, player):
    for step in move:
        board = _apply_step(board, step, player)
    return board


def choose_move(board, dice, player, network, cube=1):
    legal_moves = generate_legal_moves(board, dice, player)
    if not legal_moves:
        return None

    inputs = [encode_board(apply_move(board, m, player), player, cube) for m in legal_moves]
    batch  = torch.tensor(inputs, dtype=torch.float32, device=device)

    with torch.no_grad():
        probs = network(batch)

    weights  = torch.tensor([1., 2., 3., -1., -2., -3.], device=device)
    equities = (probs * weights).sum(dim=1)
    return legal_moves[equities.argmax().item()]


def win_rate_vs_random(net, n_games=200, cube=1):
    """Play net as WHITE against a random-moving BLACK. Returns win rate 0.0-1.0."""
    net.eval()
    wins = 0
    with torch.no_grad():
        for _ in range(n_games):
            board  = initial_board()
            player = "WHITE"
            for _ in range(10_000):
                dice  = roll_dice()
                moves = generate_legal_moves(board, dice, player)
                if moves:
                    if player == "WHITE":
                        move = choose_move(board, dice, player, net, cube)
                    else:
                        move = random.choice(moves)
                    board = apply_move(board, move, player)
                winner, _ = game_outcome(board)
                if winner is not None:
                    if winner == "WHITE":
                        wins += 1
                    break
                player = "BLACK" if player == "WHITE" else "WHITE"
    net.train()
    return wins / n_games


def train(n_games=100_000, lr=0.01, hidden_size=160, print_every=1_000, eval_every=10_000):
    """
    Self-play TD training on MPS (Apple GPU).
    Prints games/sec every print_every games.
    Plays 200 games vs a random opponent every eval_every games to measure strength.
    Start with n_games=10_000 for a quick smoke test.
    """
    net  = TDGammon(hidden_size=hidden_size, lr=lr)
    cube = 1
    t0   = time.time()

    print(f"{'Game':>10}  {'Progress':>8}  {'Games/sec':>10}  {'vs Random':>10}")
    print("-" * 48)

    for game in range(n_games):
        board  = initial_board()
        player = "WHITE"

        for _ in range(10_000):
            x = torch.tensor(encode_board(board, player, cube),
                             dtype=torch.float32, device=device)
            v = net(x)

            dice  = roll_dice()
            moves = generate_legal_moves(board, dice, player)
            if moves:
                board = apply_move(board, choose_move(board, dice, player, net, cube), player)

            winner, outcome = game_outcome(board)
            if winner is not None:
                target = torch.tensor(outcome_target(winner, outcome, player), device=device)
                net.td_update(v, target - v.detach())
                break

            next_player = "BLACK" if player == "WHITE" else "WHITE"
            with torch.no_grad():
                v_new = net(torch.tensor(encode_board(board, next_player, cube),
                                         dtype=torch.float32, device=device))
            net.td_update(v, -v_new - v.detach())
            player = next_player

        if (game + 1) % print_every == 0:
            elapsed = time.time() - t0
            gps     = (game + 1) / elapsed
            pct     = (game + 1) / n_games * 100

            if (game + 1) % eval_every == 0:
                wr = win_rate_vs_random(net)
                print(f"{game+1:>10,}  {pct:>7.1f}%  {gps:>10.1f}  {wr:>9.1%}")
            else:
                print(f"{game+1:>10,}  {pct:>7.1f}%  {gps:>10.1f}  {'—':>10}")

    print("-" * 48)
    print(f"Done. Total time: {(time.time()-t0)/60:.1f} min")
    return net


Using device: mps


In [ ]:
net = train(n_games=10_000, eval_every=1000)